# 04 Problem 1 extras: Grad-CAM + bootstrap CIs
T4 GPU (CPU works, slower). How sure are the AUROCs, and where does the ResNet look?

Refs: Grad-CAM (arXiv 1610.02391, own code in `src/gradcam.py`), bootstrap (Efron & Tibshirani 1993), deletion test (RISE, arXiv 1806.07421)

In [ ]:
# setup (same in every notebook)
import os, sys, subprocess

try:  # autoreload breaks on some Colab Python versions
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload unavailable, restart runtime after pulling changes")

REPO_URL = "https://github.com/Devanshu1503/CV_project_genimage-detection.git"
REPO_DIR = "/content/CV_project_genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run from notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()  # config dict + paths

## 1. Setup
Only saved Problem 1 outputs, nothing is retrained.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

from src import datasets as ds
from src import eda_utils as eu
from src import evaluate as ev
from src import gradcam as gc
from src import models as mdl
from src import stats as st
from src.config import generator_names, get_logger

eu.set_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("04_problem1_gradcam", P.logs)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
SEED = cfg["project"]["seed"]
NAMES = generator_names(cfg)
GEN_ORDER = [g["folder"] for g in cfg["generators"]]
TRAIN_GEN = cfg["train_generator"]
THR = cfg["evaluation"]["threshold"]
RCFG = cfg["training"]["resnet50"]
GC, BS = cfg["gradcam"], cfg["bootstrap"]
RES, FIG = P.results / "problem1_extras", P.figures / "problem1_extras"
RES.mkdir(parents=True, exist_ok=True)
MODEL_ORDER = ["ResNet-50 (fine-tuned)", "CLIP ViT-L/14 + LR", "DINOv2 ViT-B/14 + LR"]
RESNET, CLIP, DINO = MODEL_ORDER


def save(fig, name):
    eu.savefig(fig, f"04_{name}", FIG)
    plt.show()


preds = pd.read_csv(P.results / "problem1_predictions.csv")
test_preds = preds.loc[preds["split_set"] == "test"].copy()
display(test_preds.groupby(["generator", "model"]).size().unstack("model"))

## 2. Bootstrap 95% CIs for AUROC
Resample reals and fakes separately 1000 times, CI = middle 95% of the AUROCs.

In [ ]:
ci = st.bootstrap_table(test_preds, GEN_ORDER, TRAIN_GEN, n_boot=BS["n_boot"], seed=SEED, alpha=BS["alpha"])
ci.insert(2, "generator_name", ci["generator"].map(NAMES).fillna(ci["generator"]))
ev.save_table(ci, RES / "problem1_auroc_bootstrap_ci.csv")
cols = [NAMES[g] for g in GEN_ORDER] + ["mean_unseen"]
nice = (ci.assign(txt=[f"{a:.3f} [{lo:.3f}, {hi:.3f}]" for a, lo, hi in zip(ci["auroc"], ci["lo"], ci["hi"])])
          .pivot(index="model", columns="generator_name", values="txt")[cols].reindex(MODEL_ORDER))
display(nice)
fig = st.plot_ci(ci, GEN_ORDER, NAMES, MODEL_ORDER, title="Problem 1 test AUROC with 95% bootstrap CI")
save(fig, "auroc_bootstrap_ci")

### Paired bootstrap
Same resampled images for both models. `significant` = CI of the difference excludes 0.

In [ ]:
pairs = [(CLIP, RESNET), (CLIP, DINO), (RESNET, DINO)]
diffs = pd.concat([st.paired_diff_table(test_preds, a, b, GEN_ORDER, TRAIN_GEN, BS["n_boot"], SEED, BS["alpha"])
                   for a, b in pairs], ignore_index=True)
diffs.insert(1, "generator_name", diffs["generator"].map(NAMES).fillna(diffs["generator"]))
ev.save_table(diffs, RES / "problem1_auroc_paired_diffs.csv")
display(diffs[["model_a", "model_b", "generator_name", "diff", "lo", "hi", "p_boot", "significant"]].round(3))

fig, ax = plt.subplots(figsize=(9, 3.6))
for k, (a, b) in enumerate(pairs[:2]):
    d = diffs.loc[(diffs["model_a"] == a) & (diffs["model_b"] == b)].set_index("generator").reindex(GEN_ORDER + ["mean_unseen"])
    x = np.arange(len(d)) + (k - 0.5) * 0.25
    ax.errorbar(x, d["diff"], yerr=[d["diff"] - d["lo"], d["hi"] - d["diff"]], fmt="o", capsize=3,
                label=f"{a.split()[0]} - {b.split()[0]}")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(np.arange(len(d)), [NAMES.get(g, g) for g in d.index])
ax.set(ylabel="AUROC difference", title="Paired bootstrap AUROC differences (95% CI)")
ax.legend(fontsize=8)
save(fig, "auroc_paired_diffs")

## 3. Load the ResNet-50 champion
Grad-CAM on `layer4[-1]` (6x6 map at 192 px, so heatmaps are coarse).

In [ ]:
model = mdl.build_resnet50(weights=None)
ck = torch.load(P.checkpoints / "resnet50" / "best.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ck["model"])
model = model.to(DEVICE).eval()
print(f"Loaded best.pt (epoch {ck['epoch'] + 1}, val AUROC {ck['val_auroc']:.4f})")
cam = gc.GradCAM(model, gc.get_layer(model, GC["layer"]))
TF = ds.resnet_transforms(RCFG["crop_size"], train=False)  # same center crop as evaluation

index = ds.load_index(P.manifests)
rp = test_preds.loc[test_preds["model"] == RESNET].merge(
    index[["sample_id", "processed_relpath", "generator_name"]], on="sample_id", validate="1:1")
rp["pred"] = (rp["score"] >= THR).astype(int)
rp["correct"] = rp["pred"] == rp["y"]
ROOT = P.work_dir  # copy the few images we need to local disk


def run_cams(df, sign, keep_imgs=False):
    """cams, P(fake) (fp32) and optionally the shown images for the rows of df."""
    ds.stage_locally(df["processed_relpath"], P.project_root, ROOT)
    dl = DataLoader(ds.GenImageDataset(df, ROOT, TF), batch_size=GC["batch_size"], shuffle=False, num_workers=2)
    cams, probs, imgs = [], [], []
    for x, _, _ in dl:
        c, p = cam(x.to(DEVICE), sign=sign)
        cams.append(c.astype(np.float16))
        probs.append(p)
        if keep_imgs:
            imgs.append(np.stack([gc.to_uint8(t) for t in x]))
    return np.concatenate(cams), np.concatenate(probs), (np.concatenate(imgs) if keep_imgs else None)

## 4. Gallery: correct vs wrong, seen vs unseen
Most confident case per outcome. Heatmap explains the predicted class.

In [ ]:
OUTCOMES = {"fake caught": (1, True), "fake missed": (1, False), "real correct": (0, True), "real flagged": (0, False)}
picks = []
for gen in GC["gallery_generators"]:
    for name, (y, ok) in OUTCOMES.items():
        g = rp.loc[(rp["generator"] == gen) & (rp["y"] == y) & (rp["correct"] == ok)]
        g = g.assign(conf=(g["score"] - THR).abs()).nlargest(GC["n_per_cell"], "conf")
        picks.append(g.assign(outcome=name))
gal = pd.concat(picks, ignore_index=True)
display(gal.groupby(["generator_name", "outcome"]).size().unstack(fill_value=0))
gcams, gprobs, gimgs = run_cams(gal, sign="pred", keep_imgs=True)
print(f"max |P(fake) here - P(fake) in csv| = {np.abs(gprobs - gal['score']).max():.4f} (fp32 vs AMP, should be small)")

In [ ]:
n = GC["n_per_cell"]
for gen in GC["gallery_generators"]:
    fig, axes = plt.subplots(len(OUTCOMES), 2 * n, figsize=(2 * n * 1.5, len(OUTCOMES) * 1.8))
    for ax in axes.ravel():
        ax.axis("off")
    for r, name in enumerate(OUTCOMES):
        rows = gal.index[(gal["generator"] == gen) & (gal["outcome"] == name)]
        axes[r, 0].text(-0.1, 0.5, name, transform=axes[r, 0].transAxes, rotation=90, ha="right", va="center", fontsize=9)
        if len(rows) == 0:
            axes[r, 0].text(0.5, 0.5, "none", ha="center", va="center", fontsize=8)
        for k, i in enumerate(rows):
            axes[r, 2 * k].imshow(gimgs[i])
            axes[r, 2 * k].set_title(f"P(fake)={gprobs[i]:.2f}", fontsize=7)
            axes[r, 2 * k + 1].imshow(gc.overlay(gimgs[i], gcams[i].astype(np.float32)))
    seen = "seen in training" if gen == TRAIN_GEN else "unseen"
    fig.suptitle(f"Grad-CAM ResNet-50, {NAMES[gen]} ({seen})", fontweight="bold")
    fig.tight_layout()
    save(fig, f"gradcam_gallery_{NAMES[gen]}")

## 5. Mean heatmaps
100 real + 100 fake per generator, heatmap of the fake logit. Flat map = top10_mass 0.10, center_mass 0.25.

In [ ]:
k = min(GC["n_mean_per_generator"], int(rp.groupby(["generator", "y"]).size().min()))
smp = rp.groupby(["generator", "y"]).sample(n=k, random_state=SEED).reset_index(drop=True)
scams, sprobs, _ = run_cams(smp, sign="fake")
smp["pred"] = (sprobs >= THR).astype(int)
smp["outcome"] = np.select([smp["y"].eq(0), smp["pred"].eq(1)], ["real", "fake caught"], "fake missed")
print(f"{len(smp)} images ({k} real + {k} fake per generator)")

groups = ["real", "fake caught", "fake missed"]
fig, axes = plt.subplots(len(groups), len(GEN_ORDER), figsize=(len(GEN_ORDER) * 1.7, len(groups) * 1.9))
for c, gen in enumerate(GEN_ORDER):
    for r, name in enumerate(groups):
        ax = axes[r, c]
        ax.axis("off")
        sel = ((smp["generator"] == gen) & (smp["outcome"] == name)).to_numpy()
        if sel.sum():
            ax.imshow(scams[sel].astype(np.float32).mean(0), cmap="jet", vmin=0, vmax=1)
        ax.set_title(f"{NAMES[gen]}\n{name} (n={sel.sum()})", fontsize=7)
fig.suptitle("Mean Grad-CAM for the fake logit", fontweight="bold")
fig.tight_layout()
save(fig, "gradcam_mean_per_generator")

In [ ]:
cs = gc.cam_stats(scams.astype(np.float32))
smp["top10_mass"], smp["center_mass"] = cs["top_mass"], cs["center_mass"]
cam_tab = (smp.groupby(["generator_name", "outcome"])[["top10_mass", "center_mass"]].mean().round(3)
              .join(smp.groupby(["generator_name", "outcome"]).size().rename("n")))
cam_tab = cam_tab.reindex([NAMES[g] for g in GEN_ORDER], level=0)
ev.save_table(cam_tab.reset_index(), RES / "gradcam_stats.csv")
display(cam_tab)

## 6. Deletion test
Blank 20% of the caught fakes: cam region vs random box (fair baseline) vs scattered pixels.

In [ ]:
tp = smp.index[smp["outcome"] == "fake caught"].to_numpy()
dl = DataLoader(ds.GenImageDataset(smp.loc[tp], ROOT, TF), batch_size=GC["batch_size"], shuffle=False)
out, pos = [], 0
for x, _, _ in dl:
    b = len(x)
    d = gc.deletion_test(model, x.to(DEVICE), scams[tp[pos:pos + b]].astype(np.float32), GC["deletion_frac"], SEED)
    out.append(pd.DataFrame(d))
    pos += b
dele = pd.concat(out, ignore_index=True).assign(generator_name=smp.loc[tp, "generator_name"].to_numpy())
pcols = ["p_orig", "p_cam_masked", "p_box_masked", "p_pixels_masked"]
dsum = dele.groupby("generator_name")[pcols].mean()
for k in ["cam", "box", "pixels"]:
    dsum[f"drop_{k}"] = dsum["p_orig"] - dsum[f"p_{k}_masked"]
dsum["n"] = dele.groupby("generator_name").size()
dsum = dsum.reindex([NAMES[g] for g in GEN_ORDER if NAMES[g] in dsum.index])
ev.save_table(dsum.reset_index(), RES / "gradcam_deletion_test.csv")
display(dsum.round(3))
fig, ax = plt.subplots(figsize=(9, 3.4))
dsum[["drop_cam", "drop_box", "drop_pixels"]].plot.bar(ax=ax)
ax.set(ylabel="mean drop in P(fake)", xlabel="", title=f"Deletion test, caught fakes (blank {GC['deletion_frac']:.0%} of pixels)")
ax.legend(["cam region", "random box (fair baseline)", "scattered pixels"], fontsize=8)
plt.xticks(rotation=0)
save(fig, "gradcam_deletion_test")
cam.remove()

## Findings
- Bootstrap CIs: about ±0.01 on SD1.4 / SD1.5 / Wukong, up to ±0.035 on ADM / GLIDE / VQDM / BigGAN. Mean unseen: CLIP 0.839 [0.830, 0.848], ResNet-50 0.728 [0.716, 0.738], DINOv2 0.707 [0.696, 0.718], no overlap, so the Problem 1 ranking is not test-set noise.
- BigGAN: whole CI below 0.5 for ResNet-50 (0.424 [0.391, 0.462]) and DINOv2 (0.397 [0.362, 0.433]), BigGAN fakes get ranked as more real than real photos.
- Paired CLIP - ResNet: tied on SD1.4 (-0.006 [-0.013, 0.001]). ResNet better on SD1.5 (-0.007) and Wukong (-0.043). CLIP better on Midjourney +0.097, VQDM +0.122, ADM +0.133, BigGAN +0.222, GLIDE +0.256, mean unseen +0.111 [0.100, 0.123]. All significant.
- Other pairs: CLIP beats DINOv2 on all 8. ResNet vs DINOv2 is mixed, ResNet ahead on mean unseen (+0.020 [0.008, 0.033]), DINOv2 ahead on Midjourney and GLIDE, tied on VQDM and BigGAN.
- Grad-CAM: caught fakes (SD1.4/1.5, Wukong, Midjourney) give one central blob, top10_mass 0.63-0.68, center_mass 0.71-0.77. Reals are spread out (center_mass 0.19-0.28). Missed ADM / GLIDE / VQDM / BigGAN fakes look like reals (center_mass 0.25-0.39, BigGAN 0.055), no SD-style cue found at all.
- Gallery: 2 of the 4 most confident real-flagged errors on SD1.4 are smooth central surfaces (yellow bowl, cup of coffee) with the heatmap on them, so smooth clean texture seems to be part of the "fake" cue.
- Deletion test: cam region drops P(fake) ~3x more than a random box on the SD family (SD1.4 0.32 vs 0.11, SD1.5 0.31 vs 0.11, Wukong 0.34 vs 0.14), 2x on Midjourney (0.66 vs 0.33), so the heatmaps point at what the model uses. But SD1.4 fakes still sit at 0.66 with the cam region blanked, most evidence is spread over the image (texture / high frequency, fits the nb 02 FFT result).
- Scattered pixels: blanking 20% scattered pixels drops SD1.4 fakes from 0.98 to 0.12, the detector is very sensitive to high-frequency disruption.
- Caveats: 6x6 maps and object-centered images, so "center" vs "object" can't be split. ADM / GLIDE / VQDM / BigGAN have only 1-10 caught fakes, deletion numbers there are too small to read.

## Run before closing the tab
Uploads pending Drive writes, otherwise the next session may not see them.

In [ ]:
from src.colab_setup import flush_drive
flush_drive()